# Reusable Data-Checking Functions
**Project:** AI Learning Assistant Adoption
**Datasets (Mendeley Data, two survey waves of the same study):**
- Wave 1 (2023-24): Higher Education Students' Early Perceptions of ChatGPT, https://data.mendeley.com/datasets/ymg9nsn6kn/2
- Wave 2 (2024-25): Higher Education Students' Evolving Perceptions of ChatGPT, https://data.mendeley.com/datasets/nv2343nwsb/2 (CC BY 4.0)

This notebook defines two reusable functions and applies them to the columns needed for our 3 specific objectives:
1. `validate_columns()` checks that the columns we need exist.
2. `summarize_missingness()` reports how much data is missing in each column.

## 1. Load the data
We also add a `year` column (survey wave), which Objective 3 needs.

In [8]:
import re
import pandas as pd

# Original files stay untouched in data/raw/
# CHANGE the Wave 1 file name to match your download
df_w1 = pd.read_csv("rawdata(2023-2024).csv")
df_w2 = pd.read_csv("rawdata(2024-2025).csv")

# Survey wave label used in Objective 3
df_w1["year"] = "2023-24"
df_w2["year"] = "2024-25"

print("Wave 1 shape (rows, columns):", df_w1.shape)
print("Wave 2 shape (rows, columns):", df_w2.shape)

Wave 1 shape (rows, columns): (23218, 175)
Wave 2 shape (rows, columns): (22963, 181)


C:\Users\User\AppData\Local\Temp\ipykernel_19212\1039705588.py:7: DtypeWarning: Columns (2: Q3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_w2 = pd.read_csv("rawdata(2024-2025).csv")
C:\Users\User\AppData\Local\Temp\ipykernel_19212\1039705588.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_w1["year"] = "2023-24"
C:\Users\User\AppData\Local\Temp\ipykernel_19212\1039705588.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_w2["year"] = "2024-25"


## 2. Function: `validate_columns()`
Checks that every column we plan to use is present, and prints a clear PASS/FAIL report.

In [9]:
def validate_columns(df, required_columns):
    """Check that a DataFrame contains all the columns we need.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataset to check.
    required_columns : list of str
        Column names that must be present.

    Returns
    -------
    dict
        {"is_valid": bool, "missing": list, "found": list}
    """
    found = [col for col in required_columns if col in df.columns]
    missing = [col for col in required_columns if col not in df.columns]
    is_valid = len(missing) == 0

    print("COLUMN VALIDATION")
    print("-" * 40)
    print(f"Required columns : {len(required_columns)}")
    print(f"Found            : {len(found)}")
    print(f"Missing          : {len(missing)}")
    if missing:
        print(f"Missing columns  : {', '.join(missing)}")
    print("Result           :", "PASS" if is_valid else "FAIL")

    return {"is_valid": is_valid, "missing": missing, "found": found}

## 3. Function: `summarize_missingness()`
Counts missing values in every column, flags columns above a chosen threshold, and prints the worst ones.

In [10]:
def summarize_missingness(df, threshold=50.0, top_n=10):
    """Summarize missing values for every column.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataset to check.
    threshold : float
        Columns with a missing percentage above this are flagged.
    top_n : int
        How many of the most-missing columns to print.

    Returns
    -------
    pandas.DataFrame
        One row per column with missing_count, missing_percent and flagged,
        sorted from most to least missing.
    """
    total_rows = len(df)
    missing_count = df.isna().sum()
    missing_percent = (missing_count / total_rows * 100).round(2)

    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent,
        "flagged": missing_percent > threshold,
    }).sort_values("missing_percent", ascending=False)

    print("MISSINGNESS SUMMARY")
    print("-" * 40)
    print(f"Rows                        : {total_rows}")
    print(f"Columns                     : {df.shape[1]}")
    print(f"Columns with no missing     : {(missing_count == 0).sum()}")
    print(f"Columns with some missing   : {(missing_count > 0).sum()}")
    print(f"Columns above {threshold:.0f}% missing : {summary['flagged'].sum()}")
    print(f"\nTop {top_n} columns by missing percent:")
    print(summary.head(top_n).to_string())

    return summary

## 4. Helper: `expand_question_columns()`
Some survey questions are split into several columns (for example Q18 into Q18a, Q18b, ...). This small helper turns a question number into the real column names in the data. If a question has no matching column, it is kept as is, so `validate_columns()` reports it as missing. Free-text columns (ending in `_text`) are left out.

In [11]:
def expand_question_columns(df, questions):
    """Turn question numbers (e.g. "Q18") into the real column names in df.

    Example: "Q18" -> ["Q18a", "Q18b", ...] if the question is split into parts.
    A question with no matching column is kept as is, so that
    validate_columns() reports it as missing.
    """
    columns = []
    for q in questions:
        matches = [
            c for c in df.columns
            if re.match(rf"^{q}([a-z_].*)?$", str(c)) and not str(c).endswith("_text")
        ]
        columns.extend(matches if matches else [q])
    return columns

## 5. Our 3 specific objectives and the questions they need

In [18]:
objectives = {
    "Objective 1: perceived academic benefits": [
        "Q13", "Q14", "Q15", "Q18", "Q24", "Q26", "Q27", "Q35a"],
    "Objective 2: emotional experiences": [
        "Q15", "Q32", "Q33"],
    "Objective 3: AI chatbots, concerns and policy awareness": [
        "Q10", "Q13", "Q15", "Q18", "Q20", "Q22", "year"],
}

waves = {"Wave 1 (2023-24) and Wave 2 (2024-25)"}

# All questions used by at least one objective (no duplicates)
all_questions = list(dict.fromkeys(q for qs in objectives.values() for q in qs))
print("Questions needed:", all_questions)

Questions needed: ['Q13', 'Q14', 'Q15', 'Q18', 'Q24', 'Q26', 'Q27', 'Q35a', 'Q32', 'Q33', 'Q10', 'Q20', 'Q22', 'year']


## 6. Use the functions on our datasets

### 6.1 Validate the columns for each objective

In [23]:
# Store both datasets by name so they can be iterated with .items()
waves = {
    "Wave 1 (2023-24) and Wave 2 (2024-25)": df_w2,
}

for wave_name, df in waves.items():
    print("#" * 50)
    print(wave_name)
    print("#" * 50)

    for objective, questions in objectives.items():
        print(f"\n{objective}")
        validate_columns(df, expand_question_columns(df, questions))

##################################################
Wave 1 (2023-24) and Wave 2 (2024-25)
##################################################

Objective 1: perceived academic benefits
COLUMN VALIDATION
----------------------------------------
Required columns : 48
Found            : 48
Missing          : 0
Result           : PASS

Objective 2: emotional experiences
COLUMN VALIDATION
----------------------------------------
Required columns : 17
Found            : 17
Missing          : 0
Result           : PASS

Objective 3: AI chatbots, concerns and policy awareness
COLUMN VALIDATION
----------------------------------------
Required columns : 32
Found            : 32
Missing          : 0
Result           : PASS


### 6.2 Test the failure case
We deliberately add a column that does not exist (`Q99`) to show that the function reports problems correctly.

In [24]:
test_result = validate_columns(df_w2, expand_question_columns(df_w2, all_questions) + ["Q99"])

COLUMN VALIDATION
----------------------------------------
Required columns : 78
Found            : 77
Missing          : 1
Missing columns  : Q99
Result           : FAIL


### 6.3 Summarize missing values in the columns we need

In [25]:
for wave_name, df in waves.items():
    print("#" * 50)
    print(wave_name)
    print("#" * 50)
    needed = [c for c in expand_question_columns(df, all_questions) if c in df.columns]
    summarize_missingness(df[needed], threshold=50, top_n=10)
    print()

##################################################
Wave 1 (2023-24) and Wave 2 (2024-25)
##################################################
MISSINGNESS SUMMARY
----------------------------------------
Rows                        : 22963
Columns                     : 77
Columns with no missing     : 1
Columns with some missing   : 76
Columns above 50% missing : 0

Top 10 columns by missing percent:
      missing_count  missing_percent  flagged
Q32m           7241            31.53    False
Q32j           7238            31.52    False
Q32g           7239            31.52    False
Q32n           7238            31.52    False
Q32k           7233            31.50    False
Q32h           7234            31.50    False
Q32l           7234            31.50    False
Q32e           7232            31.49    False
Q32i           7230            31.49    False
Q32c           7229            31.48    False



## 7. Interpretation
Write 2 to 3 sentences here in your own words, for example:
- Which of the columns we need have the most missing values in each wave, and why? (Hint: only students who had used ChatGPT answered most questions.)
- Is the missing data similar in both waves?
- Does the missing data affect which objectives we can answer?